In [ ]:
# ==========================================
# CELL 1: Import Libraries & Environment Setup
# ==========================================

import os
import random

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    cohen_kappa_score,
    confusion_matrix,
)
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import models, transforms


# 1. Menentukan Random Seed agar hasil percobaan konsisten (reproducible)
def seed_everything(seed=42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


SEED = 42
seed_everything(SEED)

# 2. Pengaturan Device (GPU / CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Menggunakan Device: {device}")
if device.type == "cuda":
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

In [ ]:
# ==========================================
# CELL 2: Download & Loading Metadata Dataset
# ==========================================

import kagglehub

# 1. Download APTOS 2019 Blindness Detection Dataset dari Kaggle
# Jika dijalankan pertama kali, kagglehub akan mengunduh dataset ke cache lokal
dataset_path = kagglehub.competition_download("aptos2019-blindness-detection")
print(f"Path dataset: {dataset_path}")

# 2. Path ke berkas CSV dan direktori gambar
csv_path = os.path.join(dataset_path, "train.csv")
images_dir = os.path.join(dataset_path, "train_images")

# 3. Load DataFrame
df = pd.read_csv(csv_path)

# Tambahkan kolom path lengkap untuk setiap file gambar (.png)
df["image_path"] = df["id_code"].apply(lambda x: os.path.join(images_dir, f"{x}.png"))

# Mapping label numerik ke nama kelas
class_names = ["No_DR", "Mild", "Moderate", "Severe", "Proliferate_DR"]
df["class_name"] = df["diagnosis"].apply(lambda x: class_names[x])

print(f"Total Sampel: {len(df)}")
print("\nLima baris pertama dataset:")
print(df.head())

print("\nDistribusi Kelas:")
print(df["diagnosis"].value_counts().sort_index())

In [ ]:
# ==========================================
# CELL 3: Ben Graham Preprocessing & Visualisasi
# ==========================================


def crop_image_from_gray(img, tol=7):
    """
    Menghapus area hitam/gelap tidak berguna di sekitar lingkaran fundus retina.
    """
    if img.ndim == 2:
        mask = img > tol
        return img[np.ix_(mask.any(1), mask.any(0))]
    elif img.ndim == 3:
        gray_img = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
        mask = gray_img > tol

        check_column = img[:, :, 0][np.ix_(mask.any(1), mask.any(0))].shape[0]
        if check_column == 0:  # Gambar terlalu gelap / gagal dikrop
            return img
        else:
            img1 = img[:, :, 0][np.ix_(mask.any(1), mask.any(0))]
            img2 = img[:, :, 1][np.ix_(mask.any(1), mask.any(0))]
            img3 = img[:, :, 2][np.ix_(mask.any(1), mask.any(0))]
            img = np.stack([img1, img2, img3], axis=-1)
        return img


def load_ben_graham_image(path, sigmaX=30, img_size=224):
    """
    1. Memuat gambar dari path
    2. Krop area latar hitam
    3. Resize ke ukuran target (224x224)
    4. Terapkan fusi Gaussian Blur untuk meningkatkan detail pembuluh darah/lesi
    """
    image = cv2.imread(path)
    if image is None:
        raise FileNotFoundError(f"Gambar tidak ditemukan di path: {path}")

    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # 1. Crop area gelap
    image = crop_image_from_gray(image)

    # 2. Resize
    image = cv2.resize(image, (img_size, img_size))

    # 3. Ben Graham Processing
    gaussian_blur = cv2.GaussianBlur(image, (0, 0), sigmaX)
    ben_graham_img = cv2.addWeighted(image, 4, gaussian_blur, -4, 128)

    return ben_graham_img


# Visualisasi perbandingan untuk 3 sampel gambar
fig, axes = plt.subplots(3, 3, figsize=(12, 10))
sample_df = df.groupby("diagnosis").first().reset_index().head(3)

for idx, row in sample_df.iterrows():
    raw_img = cv2.imread(row["image_path"])
    raw_img = cv2.cvtColor(raw_img, cv2.COLOR_BGR2RGB)
    cropped_only = crop_image_from_gray(raw_img)
    processed_img = load_ben_graham_image(row["image_path"])

    axes[idx, 0].imshow(raw_img)
    axes[idx, 0].set_title(f"Original ({row['class_name']})")
    axes[idx, 0].axis("off")

    axes[idx, 1].imshow(cropped_only)
    axes[idx, 1].set_title("Cropped Only")
    axes[idx, 1].axis("off")

    axes[idx, 2].imshow(processed_img)
    axes[idx, 2].set_title("Ben Graham Processed")
    axes[idx, 2].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# ==========================================
# CELL 3B: Precompute Ben Graham Images
# ==========================================

from tqdm import tqdm

PROCESSED_DIR = "./processed_images_224"
os.makedirs(PROCESSED_DIR, exist_ok=True)

print("Memproses dan menyimpan seluruh gambar ke disk (Precomputing)...")

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Preprocessing Images"):
    save_path = os.path.join(PROCESSED_DIR, f"{row['id_code']}.png")

    # Hanya proses jika file belum ada
    if not os.path.exists(save_path):
        img_rgb = load_ben_graham_image(row["image_path"], img_size=224)
        # Convert RGB back to BGR for cv2.imwrite
        img_bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
        cv2.imwrite(save_path, img_bgr)

# Perbarui path di DataFrame ke direktori gambar yang sudah diproses
df["processed_path"] = df["id_code"].apply(
    lambda x: os.path.join(PROCESSED_DIR, f"{x}.png")
)
print("\nPrecompute Selesai! Gambar siap digunakan dari disk.")

In [ ]:
# ==========================================
# CELL 4 (UPDATED): Fast PyTorch Dataset
# ==========================================


class APTOSDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        # Membaca gambar yang sudah di-preprocess
        img_path = self.df.loc[idx, "processed_path"]
        label = self.df.loc[idx, "diagnosis"]

        # Langsung buka via PIL (sangat cepat)
        img = Image.open(img_path).convert("RGB")

        if self.transform:
            img = self.transform(img)

        return img, torch.tensor(label, dtype=torch.long)


# Transforms untuk Train & Val/Test tetap sama
train_transform = transforms.Compose(
    [
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.5),
        transforms.RandomRotation(degrees=15),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)

val_test_transform = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)

In [ ]:
# ==========================================
# CELL 5: Data Splitting & Weighted DataLoader
# ==========================================

# 1. Stratified Split: Train (80%) vs Temp (20%)
train_df, temp_df = train_test_split(
    df, test_size=0.20, stratify=df["diagnosis"], random_state=SEED
)

# 2. Stratified Split: Validation (10%) vs Test (10%)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df["diagnosis"], random_state=SEED
)

print(f"Jumlah sampel Train      : {len(train_df)}")
print(f"Jumlah sampel Validation : {len(val_df)}")
print(f"Jumlah sampel Test       : {len(test_df)}")

# 3. Menghitung Bobot Sampel untuk WeightedRandomSampler pada Train Set
class_counts = train_df["diagnosis"].value_counts().sort_index().values
class_weights = 1.0 / class_counts
sample_weights = [class_weights[label] for label in train_df["diagnosis"]]

# Inisialisasi Sampler
train_sampler = WeightedRandomSampler(
    weights=sample_weights, num_samples=len(sample_weights), replacement=True
)

# Menghitung Class Weights untuk Cross-Entropy Loss nanti
# Normalized weight = Total / (N_classes * Class_count)
total_samples = len(train_df)
num_classes = len(class_counts)
ce_class_weights = total_samples / (num_classes * class_counts)
ce_class_weights = torch.tensor(ce_class_weights, dtype=torch.float).to(device)

print("\nComputed Class Weights untuk Cross Entropy Loss:")
for i, w in enumerate(ce_class_weights):
    print(f"Class {i} ({class_names[i]}): {w.item():.4f}")

# 4. Inisialisasi PyTorch Datasets
train_dataset = APTOSDataset(train_df, transform=train_transform)
val_dataset = APTOSDataset(val_df, transform=val_test_transform)
test_dataset = APTOSDataset(test_df, transform=val_test_transform)

# 5. Inisialisasi DataLoaders
BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=train_sampler,  # Sampler menggantikan shuffle=True
    num_workers=2,
    pin_memory=True,
)

val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True
)

test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True
)

# Verifikasi bentuk tensor dari batch pertama
for images, labels in train_loader:
    print(f"\nBatch Images Shape: {images.shape}")
    print(f"Batch Labels Shape: {labels.shape}")
    break

In [ ]:
# ==========================================
# CELL 6: Standard DenseNet-121 Architecture
# ==========================================


def get_standard_densenet121(num_classes=5, pretrained=True):
    """
    Membangun model Standard DenseNet-121 pre-trained.
    Classifier head diganti untuk output 5 kelas DR.
    """
    weights = models.DenseNet121_Weights.DEFAULT if pretrained else None
    model = models.densenet121(weights=weights)

    # Ambil jumlah input fitur dari classifier asli (1024)
    in_features = model.classifier.in_features

    # Ganti classifier head
    model.classifier = nn.Sequential(
        nn.Linear(in_features, 512),
        nn.ReLU(),
        nn.Dropout(p=0.3),
        nn.Linear(512, num_classes),
    )
    return model


# Inisialisasi Standard Model
standard_model = get_standard_densenet121(num_classes=5, pretrained=True)
standard_model = standard_model.to(device)

print("Standard DenseNet-121 berhasil diinisialisasi.")
print("\nStruktur Classifier Head:")
print(standard_model.classifier)

In [ ]:
# ==========================================
# CELL 7: Bayesian DenseNet-121 Architecture
# ==========================================


class MCDropout(nn.Dropout):
    """
    Monte Carlo Dropout Layer:
    Memaksa fungsionalitas Dropout tetap AKTIF (training=True)
    bahkan ketika model berada dalam mode evaluasi (model.eval()).
    """

    def forward(self, x):
        return F.dropout(x, p=self.p, training=True, inplace=self.inplace)


class BayesianDenseNet121(nn.Module):
    def __init__(self, num_classes=5, pretrained=True, dropout_rate=0.3):
        super(BayesianDenseNet121, self).__init__()
        weights = models.DenseNet121_Weights.DEFAULT if pretrained else None
        self.backbone = models.densenet121(weights=weights)

        in_features = self.backbone.classifier.in_features

        # Mengganti classifier head standar dengan MC Dropout pada Fully Connected Layers
        self.backbone.classifier = nn.Sequential(
            MCDropout(p=dropout_rate),
            nn.Linear(in_features, 512),
            nn.ReLU(),
            MCDropout(p=dropout_rate),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        return self.backbone(x)


# Inisialisasi Bayesian Model
bayesian_model = BayesianDenseNet121(num_classes=5, pretrained=True, dropout_rate=0.3)
bayesian_model = bayesian_model.to(device)

print("Bayesian DenseNet-121 (MC Dropout) berhasil diinisialisasi.")
print("\nStruktur Classifier Head dengan MC Dropout:")
print(bayesian_model.backbone.classifier)

In [ ]:
# ==========================================
# CELL 8 (UPDATED): Engine dengan TQDM & Early Stopping
# ==========================================

from tqdm import tqdm


class EarlyStopping:
    """
    Menghentikan pelatihan jika skor QWK validasi tidak membaik setelah 'patience' epoch.
    """

    def __init__(self, patience=12, delta=0.001):
        self.patience = patience
        self.delta = delta
        self.counter = 0
        self.best_score = None
        self.early_stop = False

    def __call__(self, val_qwk):
        score = val_qwk
        if self.best_score is None:
            self.best_score = score
        elif score < self.best_score + self.delta:
            self.counter += 1
            print(f"  [EarlyStopping] Pasien: {self.counter}/{self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.counter = 0


def calculate_metrics(y_true, y_pred):
    """
    Menghitung Accuracy dan Quadratic Weighted Kappa (QWK)
    """
    acc = accuracy_score(y_true, y_pred)
    qwk = cohen_kappa_score(y_true, y_pred, weights="quadratic")
    return acc, qwk


def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    all_preds, all_labels = [], []

    # Progress Bar TQDM
    pbar = tqdm(dataloader, desc="Training Batch", leave=False)
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

        pbar.set_postfix({"loss": f"{loss.item():.4f}"})

    epoch_loss = running_loss / len(dataloader.dataset)
    epoch_acc, epoch_qwk = calculate_metrics(all_labels, all_preds)
    return epoch_loss, epoch_acc, epoch_qwk


def validate_one_epoch(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds, all_labels = [], []

    pbar = tqdm(dataloader, desc="Validating Batch", leave=False)
    with torch.no_grad():
        for images, labels in pbar:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    epoch_loss = running_loss / len(dataloader.dataset)
    epoch_acc, epoch_qwk = calculate_metrics(all_labels, all_preds)
    return epoch_loss, epoch_acc, epoch_qwk


def train_model(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    scheduler,
    epochs,
    model_name="model",
    patience=12,
):
    history = {
        "train_loss": [],
        "train_acc": [],
        "train_qwk": [],
        "val_loss": [],
        "val_acc": [],
        "val_qwk": [],
    }

    best_qwk = -1.0
    best_model_path = f"best_{model_name}.pth"
    early_stopping = EarlyStopping(patience=patience)

    for epoch in range(epochs):
        print(f"\nEpoch [{epoch + 1}/{epochs}]")

        train_loss, train_acc, train_qwk = train_one_epoch(
            model, train_loader, criterion, optimizer, device
        )
        val_loss, val_acc, val_qwk = validate_one_epoch(
            model, val_loader, criterion, device
        )

        if scheduler is not None:
            scheduler.step(val_loss)

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["train_qwk"].append(train_qwk)

        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        history["val_qwk"].append(val_qwk)

        print(
            f"  Train -> Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | QWK: {train_qwk:.4f}"
        )
        print(
            f"  Val   -> Loss: {val_loss:.4f} | Acc: {val_acc:.4f} | QWK: {val_qwk:.4f}"
        )

        if val_qwk > best_qwk:
            best_qwk = val_qwk
            torch.save(model.state_dict(), best_model_path)
            print(f"  --> Checkpoint Disimpan! (Val QWK: {best_qwk:.4f})")

        # Pengecekan Early Stopping
        early_stopping(val_qwk)
        if early_stopping.early_stop:
            print(
                f"\n[EARLY STOPPING TRIPPED] Pelatihan dihentikan pada epoch {epoch + 1}"
            )
            break

    print(f"Pelatihan Selesai. Best Val QWK: {best_qwk:.4f}")
    return history, best_model_path

In [ ]:
# ==========================================
# CELL 9: Pelatihan Standard DenseNet-121
# ==========================================

EPOCHS = 50
LR = 1e-4

# 1. Inisialisasi ulang model standar
standard_model = get_standard_densenet121(num_classes=5, pretrained=True).to(device)

# 2. Definisi Loss Function (menggunakan Class Weights dari Cell 5)
criterion = nn.CrossEntropyLoss(weight=ce_class_weights)

# 3. Definisi Optimizer & Scheduler
optimizer_std = torch.optim.AdamW(standard_model.parameters(), lr=LR, weight_decay=1e-4)
scheduler_std = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_std, mode="min", factor=0.5, patience=5
)

print("--- Memulai Pelatihan Standard DenseNet-121 ---")
std_history, std_best_path = train_model(
    model=standard_model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer_std,
    scheduler=scheduler_std,
    epochs=EPOCHS,
    model_name="standard_densenet121",
    patience=12,
)

In [ ]:
# ==========================================
# CELL 10: Pelatihan Bayesian DenseNet-121 (MC Dropout)
# ==========================================

# 1. Inisialisasi ulang model Bayesian
bayesian_model = BayesianDenseNet121(
    num_classes=5, pretrained=True, dropout_rate=0.3
).to(device)

# 2. Optimizer & Scheduler khusus untuk Bayesian Model
optimizer_bayes = torch.optim.AdamW(
    bayesian_model.parameters(), lr=LR, weight_decay=1e-4
)
scheduler_bayes = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_bayes, mode="min", factor=0.5, patience=5
)

print("--- Memulai Pelatihan Bayesian DenseNet-121 (MC Dropout) ---")
bayes_history, bayes_best_path = train_model(
    model=bayesian_model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer_bayes,
    scheduler=scheduler_bayes,
    epochs=EPOCHS,
    model_name="bayesian_densenet121",
    patience=12,
)

In [ ]:
# ==========================================
# CELL 11: Monte Carlo Sampling Engine
# ==========================================


def mc_predict(model, dataloader, T=30, device="cuda"):
    """
    Melakukan Monte Carlo Sampling (T forward passes) pada model Bayesian.
    Menghitung mean probabilitas, prediksi final, Predictive Entropy, dan Epistemic Variance.
    """
    model.eval()  # Layer MCDropout tetap aktif karena subclassing custom

    all_mc_probs = []
    all_labels = []

    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc=f"Monte Carlo Sampling (T={T})"):
            images = images.to(device)

            # Matriks penampung T forward passes untuk batch ini
            batch_mc_probs = []
            for _ in range(T):
                outputs = model(images)
                probs = F.softmax(outputs, dim=1)
                batch_mc_probs.append(probs.cpu().numpy())

            # Shape: (T, batch_size, 5) -> Transpose ke (batch_size, T, 5)
            batch_mc_probs = np.array(batch_mc_probs)
            batch_mc_probs = np.transpose(batch_mc_probs, (1, 0, 2))

            all_mc_probs.append(batch_mc_probs)
            all_labels.extend(labels.numpy())

    # Combine seluruh batch -> Shape: (N_samples, T, 5)
    all_mc_probs = np.concatenate(all_mc_probs, axis=0)
    all_labels = np.array(all_labels)

    # 1. Rata-rata probabilitas prediksi dari T sampel
    mean_probs = np.mean(all_mc_probs, axis=1)  # Shape: (N_samples, 5)
    preds = np.argmax(mean_probs, axis=1)

    # 2. Predictive Entropy (H = - sum(p * log(p)))
    entropy = -np.sum(mean_probs * np.log(mean_probs + 1e-12), axis=1)

    # 3. Epistemic Uncertainty (Variansi probabilitas pada kelas terprediksi)
    variance_matrix = np.var(all_mc_probs, axis=1)  # Shape: (N_samples, 5)
    pred_variance = np.array([variance_matrix[i, preds[i]] for i in range(len(preds))])

    return {
        "mean_probs": mean_probs,
        "predictions": preds,
        "true_labels": all_labels,
        "entropy": entropy,
        "variance": pred_variance,
        "raw_mc_probs": all_mc_probs,
    }


# Load bobot terbaik Bayesian Model
bayesian_model.load_state_dict(torch.load(bayes_best_path))

print("Menjalankan Monte Carlo Sampling (T=30) pada Test Set...")
bayes_test_results = mc_predict(bayesian_model, test_loader, T=30, device=device)
print("Monte Carlo Sampling Selesai.")

In [ ]:
# ==========================================
# CELL 12: Test Set Evaluation & Quantitative Comparison
# ==========================================

# 1. Evaluasi Standard Model pada Test Set
standard_model.load_state_dict(torch.load(std_best_path))
std_test_loss, std_test_acc, std_test_qwk = validate_one_epoch(
    standard_model, test_loader, criterion, device
)

# Dapatkan prediksi mentah Standard Model untuk Confusion Matrix
standard_model.eval()
std_preds, true_labels = [], []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = standard_model(images)
        preds = torch.argmax(outputs, dim=1)
        std_preds.extend(preds.cpu().numpy())
        true_labels.extend(labels.numpy())

# 2. Metrik Bayesian Model dari Monte Carlo Sampling (Cell 11)
bayes_preds = bayes_test_results["predictions"]
bayes_test_acc, bayes_test_qwk = calculate_metrics(true_labels, bayes_preds)

# Hitung NLL Loss dari mean probabilitas Bayesian
bayes_mean_probs_tensor = torch.tensor(bayes_test_results["mean_probs"])
true_labels_tensor = torch.tensor(true_labels)
bayes_test_loss = F.nll_loss(
    torch.log(bayes_mean_probs_tensor + 1e-12), true_labels_tensor
).item()

# 3. Ringkasan Performa Komparatif
metrics_comparison = pd.DataFrame(
    {
        "Model": ["Standard DenseNet-121", "Bayesian DenseNet-121 (MC Dropout)"],
        "Test Loss": [std_test_loss, bayes_test_loss],
        "Test Accuracy": [std_test_acc, bayes_test_acc],
        "Test QWK Score": [std_test_qwk, bayes_test_qwk],
    }
)

print("=== TABEL KOMPARASI PERFORMA TEST SET ===")
print(metrics_comparison.to_string(index=False))

# 4. Visualisasi Confusion Matrix Side-by-Side
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm_std = confusion_matrix(true_labels, std_preds)
cm_bayes = confusion_matrix(true_labels, bayes_preds)

sns.heatmap(
    cm_std,
    annot=True,
    fmt="d",
    cmap="Blues",
    ax=axes[0],
    xticklabels=class_names,
    yticklabels=class_names,
)
axes[0].set_title("Standard DenseNet-121")
axes[0].set_xlabel("Predicted Label")
axes[0].set_ylabel("True Label")

sns.heatmap(
    cm_bayes,
    annot=True,
    fmt="d",
    cmap="Greens",
    ax=axes[1],
    xticklabels=class_names,
    yticklabels=class_names,
)
axes[1].set_title("Bayesian DenseNet-121 (MC Dropout)")
axes[1].set_xlabel("Predicted Label")
axes[1].set_ylabel("True Label")

plt.tight_layout()
plt.show()

In [ ]:
# ==========================================
# CELL 12B: Classification Report Per Kelas
# ==========================================

print("==================================================================")
print("       CLASSIFICATION REPORT: Standard DenseNet-121")
print("==================================================================")
print(classification_report(true_labels, std_preds, target_names=class_names, digits=4))

print("\n==================================================================")
print("    CLASSIFICATION REPORT: Bayesian DenseNet-121 (MC Dropout)")
print("==================================================================")
print(
    classification_report(true_labels, bayes_preds, target_names=class_names, digits=4)
)

In [ ]:
# ==========================================
# CELL 13: Uncertainty Quantification & Visual Analysis
# ==========================================

entropy = bayes_test_results["entropy"]
variance = bayes_test_results["variance"]
bayes_preds = bayes_test_results["predictions"]
is_correct = bayes_preds == true_labels

# 1. Visualisasi Distribusi Entropy (Prediksi Benar vs Salah)
plt.figure(figsize=(9, 4))
sns.kdeplot(
    entropy[is_correct],
    label="Prediksi Benar (Correct)",
    fill=True,
    color="green",
    alpha=0.4,
)
sns.kdeplot(
    entropy[~is_correct],
    label="Prediksi Salah (Incorrect)",
    fill=True,
    color="red",
    alpha=0.4,
)
plt.title("Distribusi Predictive Entropy: Prediksi Benar vs Salah")
plt.xlabel("Predictive Entropy (Tingkat Ketidakpastian)")
plt.ylabel("Densitas Sampel")
plt.legend()
plt.show()

# 2. Mengambil Indeks Sampel dengan Uncertainty Tertinggi dan Terendah
high_unc_idx = np.argsort(entropy)[-3:]  # 3 sampel paling tidak pasti
low_unc_idx = np.argsort(entropy)[:3]  # 3 sampel paling pasti


def plot_samples(indices, title_prefix):
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    for i, idx in enumerate(indices):
        img_path = test_df.iloc[idx]["processed_path"]
        img = Image.open(img_path)

        true_cls = class_names[true_labels[idx]]
        pred_cls = class_names[bayes_preds[idx]]
        ent_val = entropy[idx]
        var_val = variance[idx]

        axes[i].imshow(img)
        axes[i].set_title(
            f"True: {true_cls} | Pred: {pred_cls}\n"
            f"Entropy: {ent_val:.3f} | Var: {var_val:.4f}",
            fontsize=10,
        )
        axes[i].axis("off")
    plt.suptitle(title_prefix, fontsize=13, fontweight="bold", y=1.05)
    plt.tight_layout()
    plt.show()


print("Visualisasi Sampel dengan Uncertainty Terendah (Model Sangat Yakin):")
plot_samples(low_unc_idx, "Low Uncertainty Examples (High Confidence)")

print(
    "\nVisualisasi Sampel dengan Uncertainty Tertinggi (Model Ragu-Ragu / Perlu Rujukan Dokter):"
)
plot_samples(
    high_unc_idx, "High Uncertainty Examples (Low Confidence / Out-of-Distribution)"
)

In [ ]:
# ==========================================
# CELL 14: Visualisasi Training History (Loss, Accuracy, QWK)
# ==========================================

epochs_std = range(1, len(std_history["train_loss"]) + 1)
epochs_bayes = range(1, len(bayes_history["train_loss"]) + 1)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# --- BARIS 1: Standard DenseNet-121 ---
# 1. Loss Plot
axes[0, 0].plot(epochs_std, std_history["train_loss"], "b-o", label="Train Loss")
axes[0, 0].plot(epochs_std, std_history["val_loss"], "r-o", label="Val Loss")
axes[0, 0].set_title("Standard DenseNet-121: Loss", fontweight="bold")
axes[0, 0].set_xlabel("Epoch")
axes[0, 0].set_ylabel("Loss")
axes[0, 0].legend()
axes[0, 0].grid(True, linestyle="--", alpha=0.6)

# 2. Accuracy Plot
axes[0, 1].plot(epochs_std, std_history["train_acc"], "b-o", label="Train Acc")
axes[0, 1].plot(epochs_std, std_history["val_acc"], "g-o", label="Val Acc")
axes[0, 1].set_title("Standard DenseNet-121: Accuracy", fontweight="bold")
axes[0, 1].set_xlabel("Epoch")
axes[0, 1].set_ylabel("Accuracy")
axes[0, 1].legend()
axes[0, 1].grid(True, linestyle="--", alpha=0.6)

# 3. QWK Plot
axes[0, 2].plot(epochs_std, std_history["train_qwk"], "b-o", label="Train QWK")
axes[0, 2].plot(epochs_std, std_history["val_qwk"], "m-o", label="Val QWK")
axes[0, 2].set_title("Standard DenseNet-121: QWK Score", fontweight="bold")
axes[0, 2].set_xlabel("Epoch")
axes[0, 2].set_ylabel("Quadratic Weighted Kappa")
axes[0, 2].legend()
axes[0, 2].grid(True, linestyle="--", alpha=0.6)

# --- BARIS 2: Bayesian DenseNet-121 (MC Dropout) ---
# 4. Loss Plot
axes[1, 0].plot(epochs_bayes, bayes_history["train_loss"], "b-s", label="Train Loss")
axes[1, 0].plot(epochs_bayes, bayes_history["val_loss"], "r-s", label="Val Loss")
axes[1, 0].set_title("Bayesian DenseNet-121: Loss", fontweight="bold")
axes[1, 0].set_xlabel("Epoch")
axes[1, 0].set_ylabel("Loss")
axes[1, 0].legend()
axes[1, 0].grid(True, linestyle="--", alpha=0.6)

# 5. Accuracy Plot
axes[1, 1].plot(epochs_bayes, bayes_history["train_acc"], "b-s", label="Train Acc")
axes[1, 1].plot(epochs_bayes, bayes_history["val_acc"], "g-s", label="Val Acc")
axes[1, 1].set_title("Bayesian DenseNet-121: Accuracy", fontweight="bold")
axes[1, 1].set_xlabel("Epoch")
axes[1, 1].set_ylabel("Accuracy")
axes[1, 1].legend()
axes[1, 1].grid(True, linestyle="--", alpha=0.6)

# 6. QWK Plot
axes[1, 2].plot(epochs_bayes, bayes_history["train_qwk"], "b-s", label="Train QWK")
axes[1, 2].plot(epochs_bayes, bayes_history["val_qwk"], "m-s", label="Val QWK")
axes[1, 2].set_title("Bayesian DenseNet-121: QWK Score", fontweight="bold")
axes[1, 2].set_xlabel("Epoch")
axes[1, 2].set_ylabel("Quadratic Weighted Kappa")
axes[1, 2].legend()
axes[1, 2].grid(True, linestyle="--", alpha=0.6)

plt.suptitle(
    "Kurva Performa Training vs Validation per Epoch",
    fontsize=16,
    fontweight="bold",
    y=1.02,
)
plt.tight_layout()
plt.show()

In [ ]:
# ==========================================
# CELL 15: Inferensi Citra Baru (Colab / Path Input)
# ==========================================

from google.colab import files


def predict_custom_image(img_path_or_bytes, std_model, bayes_model, device, T=30):
    """
    Melakukan inferensi single image untuk Standard Model vs Bayesian Model.
    Menghitung probabilitas, Predictive Entropy, dan memberikan rekomendasi rujukan.
    """
    # 1. Preprocessing citra (Ben Graham Method)
    if isinstance(img_path_or_bytes, str):
        raw_img = cv2.imread(img_path_or_bytes)
        raw_img = cv2.cvtColor(raw_img, cv2.COLOR_BGR2RGB)
        processed_img = load_ben_graham_image(img_path_or_bytes, img_size=224)
    else:
        # Jika input dari bytes upload Google Colab
        nparr = np.frombuffer(img_path_or_bytes, np.uint8)
        raw_img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
        raw_img = cv2.cvtColor(raw_img, cv2.COLOR_BGR2RGB)

        temp_path = "temp_inference_input.png"
        cv2.imwrite(temp_path, cv2.cvtColor(raw_img, cv2.COLOR_RGB2BGR))
        processed_img = load_ben_graham_image(temp_path, img_size=224)
        if os.path.exists(temp_path):
            os.remove(temp_path)

    # 2. Convert ke Tensor PyTorch
    pil_img = Image.fromarray(processed_img)
    input_tensor = val_test_transform(pil_img).unsqueeze(0).to(device)

    # 3. Prediksi Standard Model
    std_model.eval()
    with torch.no_grad():
        std_logits = std_model(input_tensor)
        std_probs = F.softmax(std_logits, dim=1).cpu().numpy()[0]
        std_pred_cls = np.argmax(std_probs)
        std_conf = std_probs[std_pred_cls]

    # 4. Prediksi Bayesian Model (Monte Carlo Sampling T=30)
    bayes_model.eval()
    mc_probs_list = []

    with torch.no_grad():
        for _ in range(T):
            bayes_logits = bayes_model(input_tensor)
            bayes_probs = F.softmax(bayes_logits, dim=1).cpu().numpy()[0]
            mc_probs_list.append(bayes_probs)

    mc_probs_arr = np.array(mc_probs_list)  # Shape: (T, 5)
    mean_bayes_probs = np.mean(mc_probs_arr, axis=0)
    bayes_pred_cls = np.argmax(mean_bayes_probs)
    bayes_conf = mean_bayes_probs[bayes_pred_cls]

    # Hitung Uncertainty Metrics
    entropy = -np.sum(mean_bayes_probs * np.log(mean_bayes_probs + 1e-12))
    variance = np.var(mc_probs_arr, axis=0)[bayes_pred_cls]

    # 5. Visualisasi Hasil Preprocessing
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].imshow(raw_img)
    axes[0].set_title("Citra Asli (Raw Input)")
    axes[0].axis("off")

    axes[1].imshow(processed_img)
    axes[1].set_title("Preprocessed (Ben Graham)")
    axes[1].axis("off")
    plt.tight_layout()
    plt.show()

    # 6. Ringkasan Diagnostik Komparatif
    print("\n" + "=" * 65)
    print("                  HASIL INFERENSI DIAGNOSIS MEDIS")
    print("=" * 65)

    print("\n[1] STANDARD DENSENET-121:")
    print(
        f"    • Prediksi Kelas     : {class_names[std_pred_cls]} (Tingkat {std_pred_cls})"
    )
    print(f"    • Tingkat Confidence : {std_conf * 100:.2f}%")

    print(f"\n[2] BAYESIAN DENSENET-121 (MC Dropout, T={T}):")
    print(
        f"    • Prediksi Kelas     : {class_names[bayes_pred_cls]} (Tingkat {bayes_pred_cls})"
    )
    print(f"    • Rata-rata Conf     : {bayes_conf * 100:.2f}%")
    print(f"    • Predictive Entropy : {entropy:.4f}")
    print(f"    • Epistemic Variance : {variance:.6f}")

    # Decision Rule Berbasis Threshold Entropy
    print("\n" + "-" * 65)
    print("REKOMENDASI DECISION ENGINE MEDIS:")
    if entropy > 0.5:
        print("⚠️  UNCERTAINTY TINGGI (Model Ragu-Ragu / Citra Kompleks)")
        print("    STATUS: PERLU RUJUKAN MANUAL KE DOKTER SPESIALIS MATA.")
    else:
        print("✅  UNCERTAINTY RENDAH (Model Sangat Yakin)")
        print(f"    STATUS: Diagnosa Otomatis Valid -> {class_names[bayes_pred_cls]}")
    print("=" * 65)


# --- JALANKAN INFERENSI INTERAKTIF ---
print("Silakan upload file gambar fundus retina baru (.png / .jpg):")
uploaded = files.upload()

for filename in uploaded.keys():
    print(f"\nMemproses citra: {filename}...")
    predict_custom_image(
        uploaded[filename], standard_model, bayesian_model, device, T=30
    )